# 08 · 12-month landmark dataset (P2)

At the 12-month visit, inputs = donor values and the 3/6/12-month visits; targets = ECD at 24/36/60 months and failure / low ECD afterwards. Split by graft (70/15/15), preprocessing fitted on training grafts only. Code: `data_prep/followup_dataset.py`.

> **Data source:** simulated post-transplant cohort calibrated to published studies (method development), because NEBSL follow-up records are not yet available. The same code runs on real follow-up data (`data_source = "nebsl"`).
>
> **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, Markdown, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
P = cfg["paths"]
OUTPUTS = Path(P["outputs"])
MODELS = Path(P["models"])
print("Cohort:", Path(P["followup_cohort"]).name, "| landmark dataset:", Path(P["followup_dataset"]).name)

In [ ]:
from recommendation.data_prep.followup_dataset import FollowupPreprocessor, leakage_checks, load
from recommendation.data_prep.followup_eda import load_cohort

table, pre = load(P["followup_dataset"], MODELS / "followup_preprocessor.json")
print(len(table), "grafts at the landmark;", len(pre.feature_names), "model inputs")
table.groupby("split").agg(grafts=("graft_id", "size"), failures=("failure_event", "sum"),
                           low_ecd_events=("low_ecd_event", "sum"))

## 1. Inputs and targets

In [ ]:
print("Inputs:", pre.numeric + pre.categorical)
table[["ECD_3m", "ECD_6m", "ECD_12m", "loss_12m", "ECD_24m", "ECD_36m", "ECD_60m"]].describe().round(1)

## 2. Leakage checks

In [ ]:
_, visits, _ = load_cohort(P["followup_cohort"])
pd.DataFrame(leakage_checks(table, pre, visits))

## 3. Missing inputs (filled with the training median and flagged)

In [ ]:
table[pre.numeric].isna().sum().loc[lambda s: s > 0]